## Data Quality Audit — Preliminary Summary

### Dataset Overview

The audit covered five datasets: products (2,000 rows), customers (800 rows), orders (3,000 rows), order items (6,219 rows), and reviews (2,500 rows).

### Key Findings

- No fully duplicated rows or duplicate primary IDs were identified.
- Missing values were concentrated in optional or conditionally populated fields, including product comparison prices, customer notes and company details, and order tracking and delivery timestamps.
- Missing product ratings were consistent with products having zero reviews.
- All checked foreign-key relationships had zero invalid references.
- Order totals matched the formula `subtotal + shipping + tax - discount` within a tolerance of 0.01.
- Order subtotals matched the sum of their associated order-item line totals.
- Product review counts matched the number of associated reviews. Product ratings were consistent with review averages rounded to one decimal place using half-up rounding.
- No negative discounts or discounts exceeding subtotals were identified.
- Order timestamps passed the preliminary chronology checks.
- Categorical values examined in products, customers, and orders showed no obvious inconsistencies.

### Items Requiring Further Investigation

- Customer-level `orders_count` and `total_spent` did not consistently match aggregates calculated from the orders dataset. The intended business definition of these metrics remains unconfirmed.
- High-value products and orders were flagged by the IQR method. These are potential statistical outliers, not confirmed errors.
- Missing values and order-status/date relationships should be interpreted using the source system's business rules where available.

### Conclusion

The initial audit found strong consistency across the checked identifiers, relationships, and financial calculations. However, passing these checks does not prove that every record is correct. Potential anomalies will be investigated further before preprocessing and exploratory data analysis.

No records were removed or modified during this audit.

In [40]:

# COMPACT CATEGORICAL AUDIT SUMMARY

columns_to_check = {
    "products": [
        "category_id", "brand", "currency", "color",
        "in_stock", "status"
    ],
    "customers": ["tier", "marketing_opt_in"],
    "orders": [
        "status", "currency", "payment_method",
        "shipping_method"
    ]
}

for dataset_name, columns in columns_to_check.items():
    df = datasets[dataset_name]

    print(f"\n--- {dataset_name.upper()} ---")

    for column in columns:
        values = df[column]
        print(
            f"{column}: "
            f"unique={values.nunique(dropna=False)}, "
            f"missing={values.isna().sum()}"
        )
        print("  ", sorted(values.dropna().astype(str).unique().tolist()))



--- PRODUCTS ---
category_id: unique=8, missing=0
   ['1', '2', '3', '4', '5', '6', '7', '8']
brand: unique=32, missing=0
   ['Basalt', 'Brisk', 'Ember', 'Fjord', 'Foxglove', 'Granite', 'Halden', 'Hearth & Co.', 'Kestrel', 'Kettleworth', 'Kinder Forge', 'Larkspur Press', 'Linden', 'Marrow', 'Meridian House', 'Motive', 'Northlight', 'Pebble', 'Pipsqueak', 'Pulse', 'Quill', 'Rootstock', 'Sonora', 'Sunhouse', 'Tandem', 'Tessel', 'Tidewater', 'Trailhead', 'Tumble', 'Vantablack', 'Verdant', 'Wobble']
currency: unique=1, missing=0
   ['USD']
color: unique=10, missing=0
   ['Black', 'Forest', 'Graphite', 'Ivory', 'Navy', 'Olive', 'Rust', 'Sand', 'Slate', 'White']
in_stock: unique=2, missing=0
   ['False', 'True']
status: unique=3, missing=0
   ['active', 'archived', 'draft']

--- CUSTOMERS ---
tier: unique=4, missing=0
   ['gold', 'platinum', 'silver', 'standard']
marketing_opt_in: unique=2, missing=0
   ['False', 'True']

--- ORDERS ---
status: unique=6, missing=0
   ['cancelled', 'delivere

In [41]:

# CATEGORICAL AND BOOLEAN VALUE AUDIT

categorical_columns = {
    "products": [
        "category_id", "brand", "currency", "color",
        "in_stock", "status"
    ],
    "customers": [
        "tier", "marketing_opt_in"
    ],
    "orders": [
        "status", "currency", "payment_method",
        "shipping_method"
    ],
    "order_items": [],
    "reviews": []
}

for dataset_name, columns in categorical_columns.items():
    df = datasets[dataset_name]

    print(f"\n{'=' * 12} {dataset_name.upper()} {'=' * 12}")

    for column in columns:
        print(f"\n{column} — unique: {df[column].nunique(dropna=False)}")
        print(df[column].value_counts(dropna=False).to_string())



============ PRODUCTS ============

category_id — unique: 8
category_id
5    261
2    260
3    260
8    253
6    251
4    246
1    244
7    225

brand — unique: 32
brand
Pulse             88
Quill             78
Meridian House    77
Linden            75
Kettleworth       72
Pebble            72
Kestrel           71
Marrow            70
Granite           69
Vantablack        67
Halden            65
Verdant           64
Northlight        64
Tessel            64
Rootstock         64
Larkspur Press    62
Wobble            62
Basalt            60
Trailhead         60
Pipsqueak         58
Ember             57
Tidewater         57
Fjord             55
Tandem            55
Foxglove          55
Tumble            53
Sunhouse          53
Brisk             53
Kinder Forge      52
Hearth & Co.      51
Motive            49
Sonora            48

currency — unique: 1
currency
USD    2000

color — unique: 10
color
Graphite    231
Sand        215
Rust        207
Navy        205
Forest      203
Olive   

In [42]:

# INSPECT ALL COLUMN NAMES

for dataset_name, df in datasets.items():
    print(f"\n{dataset_name.upper()} — {df.shape}")
    print(df.columns.tolist())



PRODUCTS — (2000, 19)
['id', 'created_at', 'updated_at', 'sku', 'name', 'slug', 'description', 'category_id', 'brand', 'price', 'compare_at_price', 'currency', 'color', 'weight_g', 'tags', 'in_stock', 'rating', 'review_count', 'status']

CUSTOMERS — (800, 13)
['id', 'created_at', 'updated_at', 'name', 'email', 'phone', 'company', 'address', 'tier', 'marketing_opt_in', 'orders_count', 'total_spent', 'notes']

ORDERS — (3000, 21)
['id', 'created_at', 'updated_at', 'number', 'customer_id', 'status', 'subtotal', 'shipping', 'tax', 'discount', 'total', 'currency', 'payment_method', 'shipping_address', 'shipping_method', 'tracking_number', 'placed_at', 'paid_at', 'shipped_at', 'delivered_at', 'notes']

ORDER_ITEMS — (6219, 10)
['id', 'created_at', 'updated_at', 'order_id', 'product_id', 'sku', 'name', 'quantity', 'unit_price', 'line_total']

REVIEWS — (2500, 10)
['id', 'created_at', 'updated_at', 'product_id', 'customer_id', 'rating', 'title', 'body', 'verified_purchase', 'helpful_votes']


In [43]:

# INSPECT CATEGORICAL VALUES

categorical_columns = {
    "products": ["category", "brand", "status"],
    "customers": ["gender", "country", "city"],
    "orders": ["status", "currency", "payment_method", "shipping_method"],
    "order_items": [],
    "reviews": [],
}

for dataset_name, columns in categorical_columns.items():
    df = datasets[dataset_name]

    print(f"\n{'=' * 15} {dataset_name.upper()} {'=' * 15}")

    for column in columns:
        if column not in df.columns:
            continue

        print(f"\n{column}:")
        print("Unique values:", df[column].nunique(dropna=False))
        print(df[column].value_counts(dropna=False).head(20).to_string())



=============== PRODUCTS ===============

brand:
Unique values: 32
brand
Pulse             88
Quill             78
Meridian House    77
Linden            75
Kettleworth       72
Pebble            72
Kestrel           71
Marrow            70
Granite           69
Vantablack        67
Halden            65
Verdant           64
Northlight        64
Tessel            64
Rootstock         64
Larkspur Press    62
Wobble            62
Basalt            60
Trailhead         60
Pipsqueak         58

status:
Unique values: 3
status
active      1789
archived     126
draft         85

=============== CUSTOMERS ===============

=============== ORDERS ===============

status:
Unique values: 6
status
delivered    2675
cancelled     173
refunded      104
shipped        32
paid            9
pending         7

currency:
Unique values: 1
currency
USD    3000

payment_method:
Unique values: 4
payment_method
card                2080
paypal               550
bank_transfer        207
cash_on_delivery     163


In [44]:

# CHECK DISCOUNT LOGIC AND ORDER TOTAL FORMULA

tol = 0.01

orders["expected_total"] = (
    orders["subtotal"]
    + orders["shipping"]
    + orders["tax"]
    - orders["discount"]
)

discount_exceeds_subtotal = (
    orders["discount"] > orders["subtotal"] + tol
)

negative_discount = orders["discount"] < 0

total_formula_mismatch = (
    (orders["total"] - orders["expected_total"]).abs() > tol
)

print("DISCOUNT & TOTAL LOGIC")
print("-" * 40)
print("Discount greater than subtotal:", discount_exceeds_subtotal.sum())
print("Negative discounts:", negative_discount.sum())
print("Total formula mismatches:", total_formula_mismatch.sum())

display(
    orders.loc[
        discount_exceeds_subtotal | negative_discount | total_formula_mismatch,
        [
            "id", "status", "subtotal", "shipping",
            "tax", "discount", "total", "expected_total"
        ]
    ].head(10)
)


DISCOUNT & TOTAL LOGIC
----------------------------------------
Discount greater than subtotal: 0
Negative discounts: 0
Total formula mismatches: 0


,id,status,subtotal,shipping,tax,discount,total,expected_total


In [45]:

# INVESTIGATE THE 10 HIGHEST-VALUE ORDERS

top_orders = orders.nlargest(10, "total")[
    ["id", "status", "subtotal", "shipping", "tax", "discount", "total"]
].copy()

top_order_items = (
    order_items.groupby("order_id")
    .agg(
        item_rows=("id", "count"),
        total_quantity=("quantity", "sum"),
        calculated_subtotal=("line_total", "sum"),
        highest_unit_price=("unit_price", "max"),
    )
    .reset_index()
)

top_orders = top_orders.merge(
    top_order_items,
    left_on="id",
    right_on="order_id",
    how="left"
)

display(
    top_orders.sort_values("total", ascending=False)
    .round(2)
    .to_string(index=False)
)


'  id    status  subtotal  shipping     tax  discount    total  order_id  item_rows  total_quantity  calculated_subtotal  highest_unit_price\n1043   shipped  16027.02      5.99 1121.89      0.00 17154.90      1043          4              13             16027.02             2842.61\n 840 delivered  15748.95      5.99 1102.43      0.00 16857.37       840          4               7             15748.95             3492.50\n 228 cancelled  14522.97      5.99 1016.61      0.00 15545.57       228          5               9             14522.97             3038.41\n 659 cancelled  14377.20     14.99 1006.40      0.00 15398.59       659          4               8             14377.20             3311.68\n2077 delivered  13578.08      5.99  950.47      0.00 14534.54      2077          3               6             13578.08             2928.11\n 597 delivered  13405.10     14.99  938.36      0.00 14358.45       597          5               6             13405.10             3209.95\n 466 deliver

In [46]:

# INSPECT THE HIGHEST POTENTIAL OUTLIERS

for name, series in numeric_checks.items():
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    upper_bound = q3 + 1.5 * iqr

    high_outliers = series[series > upper_bound].sort_values(
        ascending=False
    )

    print(f"\n{name.upper()}")
    print("-" * 40)
    print("Upper IQR bound:", round(upper_bound, 2))
    print("Highest 10 potential outliers:")
    display(high_outliers.head(10).to_frame(name="value"))



PRODUCT_PRICE
----------------------------------------
Upper IQR bound: 1268.25
Highest 10 potential outliers:


,value
1256,3492.50
588,3460.07
252,3428.30
1455,3397.76
628,3394.56
1711,3372.84
1082,3364.57
169,3351.03
922,3326.31
706,3318.20



ORDER_TOTAL
----------------------------------------
Upper IQR bound: 4038.83
Highest 10 potential outliers:


,value
90,17154.90
1270,16857.37
2324,15545.57
1586,15398.59
917,14534.54
809,14358.45
173,13188.97
813,12006.67
2012,11847.38
2258,11746.48



ORDER_SUBTOTAL
----------------------------------------
Upper IQR bound: 3837.65
Highest 10 potential outliers:


,value
90,16027.02
1270,15748.95
2324,14522.97
1586,14377.20
173,14151.87
917,13578.08
809,13405.10
807,12742.51
1411,12097.63
813,11215.59



ITEM_UNIT_PRICE
----------------------------------------
Upper IQR bound: 1216.72
Highest 10 potential outliers:


,value
5068,3492.50
1464,3492.50
1349,3492.50
1725,3492.50
3670,3460.07
1002,3460.07
2073,3460.07
6199,3460.07
3510,3428.30
3638,3428.30


In [47]:

# DETECT POTENTIAL OUTLIERS USING THE IQR METHOD

numeric_checks = {
    "product_price": products["price"],
    "order_total": orders["total"],
    "order_subtotal": orders["subtotal"],
    "item_unit_price": order_items["unit_price"],
}

outlier_summary = []

for name, series in numeric_checks.items():
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1

    lower_bound = q1 - 1.5 * iqr
    upper_bound = q3 + 1.5 * iqr

    outliers = series[
        (series < lower_bound) | (series > upper_bound)
    ]

    outlier_summary.append({
        "column": name,
        "Q1": q1,
        "Q3": q3,
        "IQR": iqr,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "outlier_count": len(outliers),
        "outlier_percentage": len(outliers) / len(series) * 100,
    })

outlier_summary = pd.DataFrame(outlier_summary)

display(outlier_summary.round(2))


,column,Q1,Q3,IQR,lower_bound,upper_bound,outlier_count,outlier_percentage
0,product_price,86.64,559.28,472.64,-622.32,1268.25,163,8.15
1,order_total,287.22,1787.86,1500.64,-1963.74,4038.83,212,7.07
2,order_subtotal,268.98,1696.45,1427.47,-1872.22,3837.65,207,6.90
3,item_unit_price,82.64,536.27,453.64,-597.82,1216.72,527,8.47


In [48]:

# INVESTIGATE EXTREME VALUES

numeric_checks = {
    "product_price": products["price"],
    "order_total": orders["total"],
    "order_subtotal": orders["subtotal"],
    "item_quantity": order_items["quantity"],
    "item_unit_price": order_items["unit_price"],
    "review_rating": reviews["rating"],
}

for name, series in numeric_checks.items():
    print(f"\n{name.upper()}")
    print("-" * 35)
    print("Min:", series.min())
    print("Median:", series.median())
    print("95th percentile:", series.quantile(0.95))
    print("99th percentile:", series.quantile(0.99))
    print("Max:", series.max())



PRODUCT_PRICE
-----------------------------------
Min: 5.14
Median: 240.72
95th percentile: 1970.3699999999974
99th percentile: 3157.9055
Max: 3492.5

ORDER_TOTAL
-----------------------------------
Min: 8.87
Median: 792.37
95th percentile: 4782.2954999999965
99th percentile: 9632.231899999972
Max: 17154.9

ORDER_SUBTOTAL
-----------------------------------
Min: 5.14
Median: 744.485
95th percentile: 4485.775
99th percentile: 8996.401099999975
Max: 16027.02

ITEM_QUANTITY
-----------------------------------
Min: 1
Median: 1.0
95th percentile: 3.0
99th percentile: 4.0
Max: 4

ITEM_UNIT_PRICE
-----------------------------------
Min: 5.14
Median: 220.56
95th percentile: 1886.01
99th percentile: 3096.404799999997
Max: 3492.5

REVIEW_RATING
-----------------------------------
Min: 1
Median: 4.0
95th percentile: 5.0
99th percentile: 5.0
Max: 5


In [49]:

# CHECK ORDER SUBTOTALS AGAINST ORDER ITEMS

items_subtotals = (
    order_items.groupby("order_id")["line_total"]
    .sum()
    .rename("calculated_subtotal")
    .reset_index()
)

order_subtotal_check = orders[
    ["id", "subtotal", "total"]
].merge(
    items_subtotals,
    left_on="id",
    right_on="order_id",
    how="left"
)

order_subtotal_check["calculated_subtotal"] = (
    order_subtotal_check["calculated_subtotal"].fillna(0)
)

order_subtotal_check["subtotal_matches"] = (
    (
        order_subtotal_check["subtotal"]
        - order_subtotal_check["calculated_subtotal"]
    ).abs() <= 0.01
)

print("ORDER SUBTOTAL CONSISTENCY")
print("-" * 40)
print("Total orders:", len(order_subtotal_check))
print("Subtotal matches:", order_subtotal_check["subtotal_matches"].sum())
print("Subtotal mismatches:", (~order_subtotal_check["subtotal_matches"]).sum())

display(
    order_subtotal_check.loc[
        ~order_subtotal_check["subtotal_matches"]
    ].head(10)
)


ORDER SUBTOTAL CONSISTENCY
----------------------------------------
Total orders: 3000
Subtotal matches: 3000
Subtotal mismatches: 0


,id,subtotal,total,order_id,calculated_subtotal,subtotal_matches


In [50]:

from decimal import Decimal, ROUND_HALF_UP

def round_half_up(value):
    if pd.isna(value):
        return np.nan
    return float(
        Decimal(str(value)).quantize(
            Decimal("0.1"),
            rounding=ROUND_HALF_UP
        )
    )

rating_check["expected_rating"] = (
    rating_check["actual_avg_rating"].apply(round_half_up)
)

no_reviews_consistent = (
    rating_check["review_count"].eq(0)
    & rating_check["rating"].isna()
    & rating_check["actual_avg_rating"].isna()
)

reviewed_products_match = (
    rating_check["rating"].notna()
    & rating_check["expected_rating"].notna()
    & (
        (rating_check["rating"] - rating_check["expected_rating"]).abs()
        <= 0.01
    )
)

rating_check["rating_consistent_final"] = (
    no_reviews_consistent | reviewed_products_match
)

print("FINAL RATING CONSISTENCY CHECK")
print("-" * 40)
print("Consistent:", rating_check["rating_consistent_final"].sum())
print("Potential mismatches:", (~rating_check["rating_consistent_final"]).sum())

display(
    rating_check.loc[
        ~rating_check["rating_consistent_final"],
        [
            "id", "rating", "review_count",
            "actual_review_count", "actual_avg_rating",
            "expected_rating"
        ]
    ]
)


FINAL RATING CONSISTENCY CHECK
----------------------------------------
Consistent: 2000
Potential mismatches: 0


,id,rating,review_count,actual_review_count,actual_avg_rating,expected_rating


In [51]:

# CORRECTED PRODUCT RATING CONSISTENCY CHECK

rating_check = product_review_check.copy()

# Round calculated averages to one decimal place
rating_check["rounded_actual_avg_rating"] = (
    rating_check["actual_avg_rating"].round(1)
)

# Products with no reviews should have no stored rating
no_reviews_both_missing = (
    rating_check["review_count"].eq(0)
    & rating_check["rating"].isna()
    & rating_check["actual_avg_rating"].isna()
)

# Products with reviews: compare stored rating to rounded average
rating_matches = (
    rating_check["rating"].notna()
    & rating_check["rounded_actual_avg_rating"].notna()
    & (
        (
            rating_check["rating"]
            - rating_check["rounded_actual_avg_rating"]
        ).abs() <= 0.01
    )
)

rating_check["rating_consistent"] = (
    no_reviews_both_missing | rating_matches
)

print("CORRECTED PRODUCT RATING CONSISTENCY")
print("-" * 45)
print("Total products:", len(rating_check))
print("Consistent:", rating_check["rating_consistent"].sum())
print("Potential mismatches:", (~rating_check["rating_consistent"]).sum())

display(
    rating_check.loc[
        ~rating_check["rating_consistent"],
        [
            "id",
            "rating",
            "review_count",
            "actual_review_count",
            "actual_avg_rating",
            "rounded_actual_avg_rating",
        ]
    ].head(15)
)


CORRECTED PRODUCT RATING CONSISTENCY
---------------------------------------------
Total products: 2000
Consistent: 1985
Potential mismatches: 15


,id,rating,review_count,actual_review_count,actual_avg_rating,rounded_actual_avg_rating
47,48,4.3,4,4,4.25,4.2
130,131,3.3,4,4,3.25,3.2
405,406,2.3,4,4,2.25,2.2
489,490,4.3,4,4,4.25,4.2
522,523,4.3,4,4,4.25,4.2
646,647,4.3,4,4,4.25,4.2
690,691,4.3,4,4,4.25,4.2
732,733,4.3,4,4,4.25,4.2
962,963,4.3,4,4,4.25,4.2
1094,1095,4.3,4,4,4.25,4.2


In [52]:

# CHECK STORED PRODUCT RATINGS AGAINST ACTUAL REVIEW AVERAGES

rating_check = product_review_check.copy()

rating_check["rating_matches"] = (
    (
        rating_check["rating"] - rating_check["actual_avg_rating"]
    ).abs() <= 0.01
).fillna(
    rating_check["rating"].isna()
    & rating_check["actual_avg_rating"].isna()
)

print("PRODUCT RATING CONSISTENCY")
print("-" * 40)
print("Total products:", len(rating_check))
print("Rating matches:", rating_check["rating_matches"].sum())
print("Rating mismatches:", (~rating_check["rating_matches"]).sum())

display(
    rating_check.loc[
        ~rating_check["rating_matches"],
        [
            "id",
            "rating",
            "review_count",
            "actual_review_count",
            "actual_avg_rating"
        ]
    ].head(10)
)


PRODUCT RATING CONSISTENCY
----------------------------------------
Total products: 2000
Rating matches: 1296
Rating mismatches: 704


,id,rating,review_count,actual_review_count,actual_avg_rating
4,5,NaN,0,0,NaN
19,20,NaN,0,0,NaN
24,25,NaN,0,0,NaN
28,29,NaN,0,0,NaN
32,33,NaN,0,0,NaN
39,40,NaN,0,0,NaN
46,47,NaN,0,0,NaN
47,48,4.3,4,4,4.250000
49,50,3.3,3,3,3.333333
50,51,NaN,0,0,NaN


In [53]:

# CHECK PRODUCT RATINGS AGAINST REVIEWS

actual_reviews = reviews.groupby("product_id").agg(
    actual_review_count=("id", "count"),
    actual_avg_rating=("rating", "mean")
).reset_index()

product_review_check = products[
    ["id", "rating", "review_count"]
].merge(
    actual_reviews,
    left_on="id",
    right_on="product_id",
    how="left"
)

product_review_check["actual_review_count"] = (
    product_review_check["actual_review_count"].fillna(0).astype(int)
)

product_review_check["count_matches"] = (
    product_review_check["review_count"]
    == product_review_check["actual_review_count"]
)

print("PRODUCT REVIEW CONSISTENCY")
print("-" * 40)
print("Total products:", len(product_review_check))
print("Review count matches:", product_review_check["count_matches"].sum())
print("Review count mismatches:", (~product_review_check["count_matches"]).sum())

display(
    product_review_check.loc[
        ~product_review_check["count_matches"]
    ].head(10)
)


PRODUCT REVIEW CONSISTENCY
----------------------------------------
Total products: 2000
Review count matches: 2000
Review count mismatches: 0


,id,rating,review_count,product_id,actual_review_count,actual_avg_rating,count_matches


In [54]:

# FINAL CUSTOMER METRICS CHECK

tol = 0.01

customer_metrics = customers[["id", "orders_count", "total_spent"]].copy()

# Calculate metrics using delivered orders
delivered = orders.loc[orders["status"] == "delivered"]

delivered_metrics = delivered.groupby("customer_id").agg(
    calculated_orders_count=("id", "count"),
    calculated_total_spent=("total", "sum")
).reset_index()

customer_metrics = customer_metrics.merge(
    delivered_metrics,
    left_on="id",
    right_on="customer_id",
    how="left"
)

customer_metrics["calculated_orders_count"] = (
    customer_metrics["calculated_orders_count"].fillna(0).astype(int)
)

customer_metrics["calculated_total_spent"] = (
    customer_metrics["calculated_total_spent"].fillna(0)
)

customer_metrics["count_matches"] = (
    customer_metrics["orders_count"]
    == customer_metrics["calculated_orders_count"]
)

customer_metrics["spent_matches"] = (
    (customer_metrics["total_spent"]
     - customer_metrics["calculated_total_spent"]).abs() <= tol
)

print("CUSTOMER METRICS SUMMARY")
print("-" * 35)
print("Total customers:", len(customer_metrics))
print("Order count matches:", customer_metrics["count_matches"].sum())
print("Spending matches:", customer_metrics["spent_matches"].sum())
print(
    "Both metrics match:",
    (
        customer_metrics["count_matches"]
        & customer_metrics["spent_matches"]
    ).sum()
)
print(
    "Neither metric matches:",
    (
        ~customer_metrics["count_matches"]
        & ~customer_metrics["spent_matches"]
    ).sum()
)


CUSTOMER METRICS SUMMARY
-----------------------------------
Total customers: 800
Order count matches: 753
Spending matches: 753
Both metrics match: 753
Neither metric matches: 47


In [55]:

print(
    customers.loc[
        customers["id"].isin(remaining_ids),
        ["id", "orders_count", "total_spent"]
    ]
    .sort_values("id")
    .to_string(index=False)
)


 id  orders_count  total_spent
169             3      3285.58
214             4      7409.11
455             5      2897.91
543             5      5422.95


In [56]:

# COMPARE STORED AND CALCULATED METRICS

remaining_ids = remaining["id"].tolist()

display(
    customers.loc[
        customers["id"].isin(remaining_ids),
        ["id", "orders_count", "total_spent"]
    ].sort_values("id")
)


,id,orders_count,total_spent
168,169,3,3285.58
213,214,4,7409.11
454,455,5,2897.91
542,543,5,5422.95


In [57]:

print("orders columns:")
print(orders.columns.tolist())

print("\nCustomer IDs being investigated:")
print(remaining["id"].tolist())

print("\nCorrectly filtered orders:")
display(
    orders.loc[
        orders["customer_id"].isin(remaining["id"]),
        ["id", "customer_id", "status", "total",
         "placed_at", "paid_at", "shipped_at", "delivered_at"]
    ]
    .sort_values(["customer_id", "placed_at"])
    .to_string(index=False)
)


orders columns:
['id', 'created_at', 'updated_at', 'number', 'customer_id', 'status', 'subtotal', 'shipping', 'tax', 'discount', 'total', 'currency', 'payment_method', 'shipping_address', 'shipping_method', 'tracking_number', 'placed_at', 'paid_at', 'shipped_at', 'delivered_at', 'notes', 'expected_total']

Customer IDs being investigated:
[169, 214, 455, 543]

Correctly filtered orders:


'  id  customer_id    status   total            placed_at              paid_at           shipped_at         delivered_at\n 811          169 delivered  361.12 2025-10-03T09:23:55Z 2025-10-03T10:49:55Z 2025-10-04T23:44:55Z 2025-10-10T02:36:55Z\n2282          169 delivered 2155.47 2026-02-10T16:21:57Z 2026-02-10T17:23:57Z 2026-02-11T21:04:57Z 2026-02-13T18:30:57Z\n1319          169  refunded 2510.55 2026-04-11T13:09:56Z 2026-04-11T14:27:56Z                  NaN                  NaN\n1151          169  refunded  211.80 2026-06-07T06:46:53Z 2026-06-07T08:15:53Z                  NaN                  NaN\n2736          169      paid  768.99 2026-08-29T16:07:05Z 2026-08-29T16:35:05Z                  NaN                  NaN\n1225          214 delivered  416.30 2025-10-15T01:16:51Z 2025-10-15T02:08:51Z 2025-10-16T14:51:51Z 2025-10-20T04:29:51Z\n2966          214 delivered   72.36 2025-11-18T09:42:46Z 2025-11-18T10:02:46Z 2025-11-19T11:52:46Z 2025-11-20T19:37:46Z\n 347          214 delivered 533

In [58]:

# INSPECT ALL ORDERS FOR THE 4 CUSTOMERS

remaining_ids = remaining["id"].tolist()

columns_to_show = [
    "id",
    "customer_id",
    "status",
    "total",
    "placed_at",
    "paid_at",
    "shipped_at",
    "delivered_at",
]

display(
    orders.loc[
        orders["customer_id"].isin(remaining_ids),
        columns_to_show
    ].sort_values(["customer_id", "placed_at"])
)


,id,customer_id,status,total,placed_at,paid_at,shipped_at,delivered_at
2742,811,169,delivered,361.12,2025-10-03T09:23:55Z,2025-10-03T10:49:55Z,2025-10-04T23:44:55Z,2025-10-10T02:36:55Z
1684,2282,169,delivered,2155.47,2026-02-10T16:21:57Z,2026-02-10T17:23:57Z,2026-02-11T21:04:57Z,2026-02-13T18:30:57Z
1135,1319,169,refunded,2510.55,2026-04-11T13:09:56Z,2026-04-11T14:27:56Z,NaN,NaN
735,1151,169,refunded,211.80,2026-06-07T06:46:53Z,2026-06-07T08:15:53Z,NaN,NaN
21,2736,169,paid,768.99,2026-08-29T16:07:05Z,2026-08-29T16:35:05Z,NaN,NaN
2653,1225,214,delivered,416.30,2025-10-15T01:16:51Z,2025-10-15T02:08:51Z,2025-10-16T14:51:51Z,2025-10-20T04:29:51Z
2408,2966,214,delivered,72.36,2025-11-18T09:42:46Z,2025-11-18T10:02:46Z,2025-11-19T11:52:46Z,2025-11-20T19:37:46Z
1192,347,214,delivered,5337.90,2026-04-06T03:24:50Z,2026-04-06T04:18:50Z,2026-04-07T23:40:50Z,2026-04-09T18:56:50Z
246,2227,214,refunded,187.55,2026-08-07T00:15:33Z,2026-08-07T01:41:33Z,NaN,NaN
1,1601,214,paid,1582.55,2026-09-01T09:42:53Z,2026-09-01T10:05:53Z,NaN,NaN


In [59]:

# INVESTIGATE THE 4 REMAINING CUSTOMERS

remaining = analysis.loc[
    analysis["consistency_group"] == "Neither rule matches"
].copy()

remaining["count_diff_vs_delivered"] = (
    remaining["orders_count"] - remaining["delivered_orders"]
)

remaining["spent_diff_vs_delivered"] = (
    remaining["total_spent"] - remaining["delivered_spent"]
)

remaining["count_diff_vs_non_cancelled"] = (
    remaining["orders_count"] - remaining["non_cancelled_orders"]
)

remaining["spent_diff_vs_non_cancelled"] = (
    remaining["total_spent"] - remaining["non_cancelled_spent"]
)

display(
    remaining[
        [
            "id",
            "orders_count",
            "total_spent",
            "delivered_orders",
            "delivered_spent",
            "non_cancelled_orders",
            "non_cancelled_spent",
            "count_diff_vs_delivered",
            "spent_diff_vs_delivered",
            "count_diff_vs_non_cancelled",
            "spent_diff_vs_non_cancelled",
        ]
    ]
)


,id,orders_count,total_spent,delivered_orders,delivered_spent,non_cancelled_orders,non_cancelled_spent,count_diff_vs_delivered,spent_diff_vs_delivered,count_diff_vs_non_cancelled,spent_diff_vs_non_cancelled
168,169,3,3285.58,2.0,2516.59,5.0,6007.93,1.0,768.99,-2.0,-2722.35
213,214,4,7409.11,3.0,5826.56,5.0,7596.66,1.0,1582.55,-1.0,-187.55
454,455,5,2897.91,4.0,1946.96,6.0,3062.12,1.0,950.95,-1.0,-164.21
542,543,5,5422.95,4.0,3560.50,6.0,5833.60,1.0,1862.45,-1.0,-410.65


In [60]:

# CLASSIFY CUSTOMER METRIC CONSISTENCY

tol = 0.01

analysis = detailed.copy()

analysis["count_matches_non_cancelled"] = (
    analysis["orders_count"]
    == analysis["non_cancelled_orders"]
)

analysis["spent_matches_non_cancelled"] = (
    (
        analysis["total_spent"]
        - analysis["non_cancelled_spent"]
    ).abs() <= tol
)

analysis["count_matches_delivered"] = (
    analysis["orders_count"]
    == analysis["delivered_orders"]
)

analysis["spent_matches_delivered"] = (
    (
        analysis["total_spent"]
        - analysis["delivered_spent"]
    ).abs() <= tol
)

def classify_customer(row):
    if (
        row["count_matches_delivered"]
        and row["spent_matches_delivered"]
    ):
        return "Both match delivered"

    if (
        row["count_matches_non_cancelled"]
        and row["spent_matches_non_cancelled"]
    ):
        return "Both match non-cancelled"

    if (
        row["count_matches_delivered"]
        or row["spent_matches_delivered"]
        or row["count_matches_non_cancelled"]
        or row["spent_matches_non_cancelled"]
    ):
        return "Partial match"

    return "Neither rule matches"

analysis["consistency_group"] = analysis.apply(
    classify_customer,
    axis=1
)

print("CUSTOMER CONSISTENCY GROUPS")
print("-" * 45)

display(
    analysis["consistency_group"]
    .value_counts()
    .rename_axis("group")
    .reset_index(name="customers")
)


CUSTOMER CONSISTENCY GROUPS
---------------------------------------------


,group,customers
0,Both match delivered,753
1,Both match non-cancelled,43
2,Neither rule matches,4


In [61]:

# INVESTIGATE CUSTOMERS WITH METRIC MISMATCHES

tol = 0.01

mismatch_ids = detailed.loc[
    ~detailed["count_matches_delivered"]
    | ~detailed["spent_matches_delivered"],
    "id"
]

mismatch_orders = orders.loc[
    orders["customer_id"].isin(mismatch_ids)
].copy()

print("Orders belonging to mismatched customers:", len(mismatch_orders))

print("\nOrder status distribution for these customers:")
display(
    mismatch_orders["status"]
    .value_counts()
    .rename_axis("status")
    .reset_index(name="count")
)

print("\nOrder-level details for the first 10 mismatched customers:")
display(
    mismatch_orders.loc[
        mismatch_orders["customer_id"].isin(mismatch_ids.head(10)),
        [
            "customer_id",
            "id",
            "status",
            "subtotal",
            "shipping",
            "tax",
            "discount",
            "total",
        ],
    ].sort_values(["customer_id", "id"])
)



Orders belonging to mismatched customers: 236

Order status distribution for these customers:


,status,count
0,delivered,173
1,shipped,32
2,cancelled,10
3,paid,9
4,pending,7
5,refunded,5



Order-level details for the first 10 mismatched customers:


,customer_id,id,status,subtotal,shipping,tax,discount,total
2110,27,20,delivered,1071.69,29.99,75.02,0.00,1176.70
2891,27,149,delivered,3221.96,5.99,225.54,0.00,3453.49
1733,27,685,delivered,6999.29,5.99,489.95,0.00,7495.23
2019,27,832,delivered,3592.39,14.99,251.47,0.00,3858.85
2513,27,1069,delivered,1044.54,5.99,66.54,94.01,1023.06
0,27,1889,pending,914.18,5.99,56.95,100.56,876.56
966,27,2376,delivered,669.23,14.99,46.85,0.00,731.07
2346,27,2568,delivered,40.76,5.99,2.85,0.00,49.60
813,39,43,delivered,11215.59,5.99,785.09,0.00,12006.67
728,39,1005,delivered,15.61,5.99,1.09,0.00,22.69


In [62]:

# CUSTOMER METRICS: DETAILED MISMATCH ANALYSIS

tol = 0.01

count_diff = (
    customer_spending["total_spent"]
    - customers["total_spent"]
).abs() <= tol

# Reuse the existing investigation table
count_matches_delivered = (
    investigation["orders_count"]
    == investigation["delivered_orders"]
)

count_matches_non_cancelled = (
    investigation["orders_count"]
    == investigation["non_cancelled_orders"]
)

spent_matches_delivered = (
    customer_spending["total_spent"]
    - customer_spending["delivered_spent"]
).abs() <= tol

spent_matches_non_cancelled = (
    customer_spending["total_spent"]
    - customer_spending["non_cancelled_spent"]
).abs() <= tol

print("CUSTOMER METRIC PATTERNS")
print("-" * 45)

print(
    "Order count matches delivered orders:",
    int(count_matches_delivered.sum())
)

print(
    "Order count matches non-cancelled orders:",
    int(count_matches_non_cancelled.sum())
)

print(
    "Total spent matches delivered spending:",
    int(spent_matches_delivered.sum())
)

print(
    "Total spent matches non-cancelled spending:",
    int(spent_matches_non_cancelled.sum())
)

# Customers where either stored metric differs from delivered-based metrics
detailed = investigation[
    ["id", "orders_count", "total_spent",
     "delivered_orders", "non_cancelled_orders"]
].merge(
    customer_spending[
        ["id", "delivered_spent", "non_cancelled_spent"]
    ],
    on="id",
    how="left"
)

detailed["count_matches_delivered"] = (
    detailed["orders_count"] == detailed["delivered_orders"]
)

detailed["spent_matches_delivered"] = (
    (detailed["total_spent"] - detailed["delivered_spent"]).abs() <= tol
)

print("\nCustomers whose stored metrics do not match delivered metrics:")
display(
    detailed.loc[
        ~detailed["count_matches_delivered"]
        | ~detailed["spent_matches_delivered"]
    ].head(20)
)


CUSTOMER METRIC PATTERNS
---------------------------------------------
Order count matches delivered orders: 753
Order count matches non-cancelled orders: 703
Total spent matches delivered spending: 753
Total spent matches non-cancelled spending: 703

Customers whose stored metrics do not match delivered metrics:


,id,orders_count,total_spent,delivered_orders,non_cancelled_orders,delivered_spent,non_cancelled_spent,count_matches_delivered,spent_matches_delivered
26,27,8,18664.56,7.0,8.0,17788.00,18664.56,False,False
38,39,6,15482.64,5.0,6.0,14216.91,15482.64,False,False
40,41,4,13097.50,3.0,4.0,12354.77,13097.50,False,False
61,62,5,4873.13,4.0,5.0,4201.80,4873.13,False,False
71,72,4,1585.22,3.0,4.0,1123.78,1585.22,False,False
78,79,2,5426.87,1.0,2.0,1737.69,5426.87,False,False
122,123,2,1229.22,1.0,2.0,461.87,1229.22,False,False
126,127,6,5543.32,5.0,6.0,3266.24,5543.32,False,False
127,128,3,3245.70,2.0,3.0,1960.75,3245.70,False,False
130,131,2,921.26,1.0,2.0,259.01,921.26,False,False


In [63]:

# INVESTIGATE CUSTOMER TOTAL SPENT

customer_spending = customers[
    ["id", "total_spent"]
].copy()

# Sum of all order totals
all_spending = orders.groupby("customer_id")["total"].sum()

# Exclude cancelled orders
non_cancelled_spending = (
    orders.loc[orders["status"] != "cancelled"]
    .groupby("customer_id")["total"]
    .sum()
)

# Delivered orders only
delivered_spending = (
    orders.loc[orders["status"] == "delivered"]
    .groupby("customer_id")["total"]
    .sum()
)

customer_spending["all_orders_spent"] = (
    customer_spending["id"]
    .map(all_spending)
    .fillna(0)
)

customer_spending["non_cancelled_spent"] = (
    customer_spending["id"]
    .map(non_cancelled_spending)
    .fillna(0)
)

customer_spending["delivered_spent"] = (
    customer_spending["id"]
    .map(delivered_spending)
    .fillna(0)
)

tolerance = 0.01

print("CUSTOMER TOTAL SPENT INVESTIGATION")
print("-" * 45)

for column in [
    "all_orders_spent",
    "non_cancelled_spent",
    "delivered_spent",
]:
    matches = (
        customer_spending["total_spent"]
        - customer_spending[column]
    ).abs() <= tolerance

    print(f"Matches {column}: {int(matches.sum())} / {len(customers)}")

print("\nExamples where total_spent differs from all order totals:")

display(
    customer_spending.loc[
        (
            customer_spending["total_spent"]
            - customer_spending["all_orders_spent"]
        ).abs() > tolerance
    ].head(15)
)


CUSTOMER TOTAL SPENT INVESTIGATION
---------------------------------------------
Matches all_orders_spent: 566 / 800
Matches non_cancelled_spent: 703 / 800
Matches delivered_spent: 753 / 800

Examples where total_spent differs from all order totals:


,id,total_spent,all_orders_spent,non_cancelled_spent,delivered_spent
0,1,5419.75,8163.84,5419.75,5419.75
4,5,3787.59,6204.91,3787.59,3787.59
5,6,0.00,308.56,308.56,0.00
7,8,1765.24,2091.88,2091.88,1765.24
9,10,5882.65,6678.86,6678.86,5882.65
14,15,809.74,3079.56,809.74,809.74
15,16,5867.66,5947.97,5947.97,5867.66
21,22,1299.31,1456.93,1299.31,1299.31
23,24,17667.64,21164.69,17745.91,17667.64
32,33,1823.42,2176.40,1823.42,1823.42


In [64]:

# INVESTIGATE CUSTOMER METRIC MISMATCHES

# Count orders by customer and status
status_counts = pd.crosstab(
    orders["customer_id"],
    orders["status"]
)

# Total order count for each customer
all_order_counts = orders.groupby("customer_id").size()

# Count only non-cancelled orders
non_cancelled_counts = (
    orders.loc[orders["status"] != "cancelled"]
    .groupby("customer_id")
    .size()
)

# Count only delivered orders
delivered_counts = (
    orders.loc[orders["status"] == "delivered"]
    .groupby("customer_id")
    .size()
)

investigation = customers[
    ["id", "orders_count", "total_spent"]
].copy()

investigation["all_orders"] = (
    investigation["id"].map(all_order_counts).fillna(0)
)

investigation["non_cancelled_orders"] = (
    investigation["id"].map(non_cancelled_counts).fillna(0)
)

investigation["delivered_orders"] = (
    investigation["id"].map(delivered_counts).fillna(0)
)

print("CUSTOMER ORDER COUNT INVESTIGATION")
print("-" * 45)

print(
    "Matches all orders:",
    int((investigation["orders_count"] == investigation["all_orders"]).sum())
)

print(
    "Matches non-cancelled orders:",
    int((investigation["orders_count"] == investigation["non_cancelled_orders"]).sum())
)

print(
    "Matches delivered orders:",
    int((investigation["orders_count"] == investigation["delivered_orders"]).sum())
)

print("\nExamples:")
display(
    investigation.loc[
        investigation["orders_count"] != investigation["all_orders"]
    ].head(15)
)

print("\nOrder status counts:")
display(status_counts.sum().rename("order_count_by_status").to_frame())


CUSTOMER ORDER COUNT INVESTIGATION
---------------------------------------------
Matches all orders: 566
Matches non-cancelled orders: 703
Matches delivered orders: 753

Examples:


,id,orders_count,total_spent,all_orders,non_cancelled_orders,delivered_orders
0,1,4,5419.75,6.0,4.0,4.0
4,5,3,3787.59,4.0,3.0,3.0
5,6,0,0.00,1.0,1.0,0.0
7,8,2,1765.24,3.0,3.0,2.0
9,10,4,5882.65,5.0,5.0,4.0
14,15,2,809.74,3.0,2.0,2.0
15,16,4,5867.66,5.0,5.0,4.0
21,22,3,1299.31,4.0,3.0,3.0
23,24,5,17667.64,7.0,6.0,5.0
32,33,3,1823.42,4.0,3.0,3.0



Order status counts:


,order_count_by_status
status,
cancelled,173
delivered,2675
paid,9
pending,7
refunded,104
shipped,32


In [65]:
# CUSTOMER METRICS VALIDATION

actual_orders = orders.groupby("customer_id").size()

actual_spent = (
    orders.groupby("customer_id")["total"].sum()
)

customer_check = customers[
    ["id", "orders_count", "total_spent"]
].copy()

customer_check["actual_orders_count"] = (
    customer_check["id"]
    .map(actual_orders)
    .fillna(0)
    .astype(int)
)

customer_check["actual_total_spent"] = (
    customer_check["id"]
    .map(actual_spent)
    .fillna(0)
)

print("CUSTOMER METRICS VALIDATION")
print("-" * 45)

print(
    "Order count mismatches:",
    int(
        (
            customer_check["orders_count"]
            != customer_check["actual_orders_count"]
        ).sum()
    )
)

print(
    "Total spent mismatches:",
    int(
        (
            (
                customer_check["total_spent"]
                - customer_check["actual_total_spent"]
            ).abs() > 0.01
        ).sum()
    )
)

print("\nOrder count mismatch examples:")

display(
    customer_check.loc[
        customer_check["orders_count"]
        != customer_check["actual_orders_count"]
    ].head(10)
)

print("\nTotal spent mismatch examples:")

display(
    customer_check.loc[
        (
            customer_check["total_spent"]
            - customer_check["actual_total_spent"]
        ).abs() > 0.01
    ].head(10)
)

CUSTOMER METRICS VALIDATION
---------------------------------------------
Order count mismatches: 234
Total spent mismatches: 234

Order count mismatch examples:


,id,orders_count,total_spent,actual_orders_count,actual_total_spent
0,1,4,5419.75,6,8163.84
4,5,3,3787.59,4,6204.91
5,6,0,0.00,1,308.56
7,8,2,1765.24,3,2091.88
9,10,4,5882.65,5,6678.86
14,15,2,809.74,3,3079.56
15,16,4,5867.66,5,5947.97
21,22,3,1299.31,4,1456.93
23,24,5,17667.64,7,21164.69
32,33,3,1823.42,4,2176.40



Total spent mismatch examples:


,id,orders_count,total_spent,actual_orders_count,actual_total_spent
0,1,4,5419.75,6,8163.84
4,5,3,3787.59,4,6204.91
5,6,0,0.00,1,308.56
7,8,2,1765.24,3,2091.88
9,10,4,5882.65,5,6678.86
14,15,2,809.74,3,3079.56
15,16,4,5867.66,5,5947.97
21,22,3,1299.31,4,1456.93
23,24,5,17667.64,7,21164.69
32,33,3,1823.42,4,2176.40


In [66]:
# STATUS-DATE CONTRADICTIONS

rules = {
    "pending_has_paid_date": (
        orders["status"].eq("pending")
        & order_dates["paid_at"].notna()
    ),
    "pending_has_shipped_date": (
        orders["status"].eq("pending")
        & order_dates["shipped_at"].notna()
    ),
    "pending_has_delivered_date": (
        orders["status"].eq("pending")
        & order_dates["delivered_at"].notna()
    ),
    "cancelled_has_shipped_date": (
        orders["status"].eq("cancelled")
        & order_dates["shipped_at"].notna()
    ),
    "cancelled_has_delivered_date": (
        orders["status"].eq("cancelled")
        & order_dates["delivered_at"].notna()
    ),
    "shipped_has_delivered_date": (
        orders["status"].eq("shipped")
        & order_dates["delivered_at"].notna()
    ),
    "refunded_missing_paid_date": (
        orders["status"].eq("refunded")
        & order_dates["paid_at"].isna()
    ),
}

print("STATUS-DATE CONTRADICTIONS")
print("-" * 45)

for rule_name, condition in rules.items():
    print(f"{rule_name}: {int(condition.sum())}")

    if condition.any():
        display(
            orders.loc[
                condition,
                ["id", "status"] + date_columns
            ].head(10)
        )

STATUS-DATE CONTRADICTIONS
---------------------------------------------
pending_has_paid_date: 0
pending_has_shipped_date: 0
pending_has_delivered_date: 0
cancelled_has_shipped_date: 0
cancelled_has_delivered_date: 0
shipped_has_delivered_date: 0
refunded_missing_paid_date: 0


In [67]:
# STATUS-DATE BUSINESS RULE VALIDATION

status_rules = {
    "delivered": ["placed_at", "paid_at", "shipped_at", "delivered_at"],
    "shipped": ["placed_at", "paid_at", "shipped_at"],
    "paid": ["placed_at", "paid_at"],
    "pending": ["placed_at"],
    "cancelled": ["placed_at"],
    "refunded": ["placed_at"],
}

print("STATUS-DATE VALIDATION")
print("-" * 45)

for status, required_dates in status_rules.items():
    mask = orders["status"].eq(status)
    subset = order_dates.loc[mask]

    if subset.empty:
        continue

    missing_required = subset[required_dates].isna().any(axis=1)

    print(f"\nStatus: {status}")
    print("Orders:", int(mask.sum()))
    print(
        "Orders missing expected dates:",
        int(missing_required.sum())
    )

    if missing_required.any():
        display(
            orders.loc[
                subset.index[missing_required],
                ["id", "status"] + required_dates
            ].head(10)
        )

STATUS-DATE VALIDATION
---------------------------------------------

Status: delivered
Orders: 2675
Orders missing expected dates: 0

Status: shipped
Orders: 32
Orders missing expected dates: 0

Status: paid
Orders: 9
Orders missing expected dates: 0

Status: pending
Orders: 7
Orders missing expected dates: 0

Status: cancelled
Orders: 173
Orders missing expected dates: 0

Status: refunded
Orders: 104
Orders missing expected dates: 0


In [68]:
# ORDER STATUS ANALYSIS

print("ORDER STATUS DISTRIBUTION")
print("-" * 40)

display(
    orders["status"]
    .value_counts(dropna=False)
    .rename_axis("status")
    .reset_index(name="count")
)

print("\nSTATUS VS DATE AVAILABILITY")
print("-" * 40)

date_columns = [
    "placed_at",
    "paid_at",
    "shipped_at",
    "delivered_at",
]

order_dates = orders[date_columns].apply(
    pd.to_datetime,
    errors="coerce",
    utc=True
)

for status in sorted(orders["status"].dropna().unique()):
    subset = orders.loc[orders["status"] == status]
    date_subset = order_dates.loc[subset.index]

    print(f"\nStatus: {status}")
    print("Orders:", len(subset))

    for column in date_columns:
        print(
            f"{column} missing:",
            int(date_subset[column].isna().sum())
        )

ORDER STATUS DISTRIBUTION
----------------------------------------


,status,count
0,delivered,2675
1,cancelled,173
2,refunded,104
3,shipped,32
4,paid,9
5,pending,7



STATUS VS DATE AVAILABILITY
----------------------------------------

Status: cancelled
Orders: 173
placed_at missing: 0
paid_at missing: 0
shipped_at missing: 173
delivered_at missing: 173

Status: delivered
Orders: 2675
placed_at missing: 0
paid_at missing: 0
shipped_at missing: 0
delivered_at missing: 0

Status: paid
Orders: 9
placed_at missing: 0
paid_at missing: 0
shipped_at missing: 9
delivered_at missing: 9

Status: pending
Orders: 7
placed_at missing: 0
paid_at missing: 7
shipped_at missing: 7
delivered_at missing: 7

Status: refunded
Orders: 104
placed_at missing: 0
paid_at missing: 0
shipped_at missing: 104
delivered_at missing: 104

Status: shipped
Orders: 32
placed_at missing: 0
paid_at missing: 0
shipped_at missing: 0
delivered_at missing: 32


In [69]:
# ORDER FINANCIAL VALIDATION

calculated_total = (
    orders["subtotal"]
    + orders["shipping"]
    + orders["tax"]
    - orders["discount"]
)

difference = (
    orders["total"] - calculated_total
).abs()

print("ORDER FINANCIAL VALIDATION")
print("-" * 45)

print("Total orders:", len(orders))

print(
    "Negative subtotal:",
    int((orders["subtotal"] < 0).sum())
)

print(
    "Negative shipping:",
    int((orders["shipping"] < 0).sum())
)

print(
    "Negative tax:",
    int((orders["tax"] < 0).sum())
)

print(
    "Negative discount:",
    int((orders["discount"] < 0).sum())
)

print(
    "Negative total:",
    int((orders["total"] < 0).sum())
)

print(
    "Total mismatches:",
    int((difference > 0.01).sum())
)

print("\nMismatch examples:")

display(
    orders.loc[
        difference > 0.01,
        [
            "id",
            "subtotal",
            "shipping",
            "tax",
            "discount",
            "total",
        ],
    ].head(10)
)

ORDER FINANCIAL VALIDATION
---------------------------------------------
Total orders: 3000
Negative subtotal: 0
Negative shipping: 0
Negative tax: 0
Negative discount: 0
Negative total: 0
Total mismatches: 0

Mismatch examples:


,id,subtotal,shipping,tax,discount,total


In [70]:
# FOREIGN KEY VALIDATION

checks = {
    "orders.customer_id": (
        orders["customer_id"],
        customers["id"]
    ),
    "order_items.order_id": (
        order_items["order_id"],
        orders["id"]
    ),
    "order_items.product_id": (
        order_items["product_id"],
        products["id"]
    ),
    "reviews.product_id": (
        reviews["product_id"],
        products["id"]
    ),
    "reviews.customer_id": (
        reviews["customer_id"],
        customers["id"]
    ),
}

print("FOREIGN KEY VALIDATION")
print("-" * 50)

for relationship, (child_values, parent_ids) in checks.items():
    invalid = ~child_values.isin(parent_ids)

    print(f"\n{relationship}")
    print("Invalid references:", int(invalid.sum()))

    if invalid.any():
        print("Examples:")
        display(child_values[invalid].head(10))

FOREIGN KEY VALIDATION
--------------------------------------------------

orders.customer_id
Invalid references: 0

order_items.order_id
Invalid references: 0

order_items.product_id
Invalid references: 0

reviews.product_id
Invalid references: 0

reviews.customer_id
Invalid references: 0


In [71]:

# ORDER ITEMS: CALCULATION & VALUE VALIDATION

calculated_total = (
    order_items["quantity"] * order_items["unit_price"]
)

difference = (
    order_items["line_total"] - calculated_total
).abs()

print("ORDER ITEM VALIDATION")
print("-" * 45)

print("Total rows:", len(order_items))

print(
    "Quantity equal to zero:",
    int((order_items["quantity"] == 0).sum())
)

print(
    "Negative quantity:",
    int((order_items["quantity"] < 0).sum())
)

print(
    "Negative unit price:",
    int((order_items["unit_price"] < 0).sum())
)

print(
    "Zero unit price:",
    int((order_items["unit_price"] == 0).sum())
)

print(
    "Negative line total:",
    int((order_items["line_total"] < 0).sum())
)

print(
    "Line total mismatches:",
    int((difference > 0.01).sum())
)

print("\nExamples of mismatches:")

display(
    order_items.loc[
        difference > 0.01,
        [
            "id",
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ],
    ].head(10)
)


ORDER ITEM VALIDATION
---------------------------------------------
Total rows: 6219
Quantity equal to zero: 0
Negative quantity: 0
Negative unit price: 0
Zero unit price: 0
Negative line total: 0
Line total mismatches: 0

Examples of mismatches:


,id,order_id,product_id,quantity,unit_price,line_total


In [72]:

# ORDER DATE QUALITY CHECK

date_columns = [
    "placed_at",
    "paid_at",
    "shipped_at",
    "delivered_at",
]

# Convert dates for analysis only; original data remains unchanged
order_dates = orders[date_columns].apply(
    pd.to_datetime,
    errors="coerce",
    utc=True,
)

print("DATE PARSING")
print("-" * 40)

for column in date_columns:
    invalid = (
        orders[column].notna()
        & order_dates[column].isna()
    ).sum()

    print(f"{column}: invalid dates = {int(invalid)}")


print("\nCHRONOLOGICAL VALIDATION")
print("-" * 40)

checks = {
    "Paid before order": ("paid_at", "placed_at"),
    "Shipped before order": ("shipped_at", "placed_at"),
    "Delivered before order": ("delivered_at", "placed_at"),
    "Delivered before shipped": ("delivered_at", "shipped_at"),
    "Delivered before paid": ("delivered_at", "paid_at"),
}

for label, (first, second) in checks.items():
    valid_pair = (
        order_dates[first].notna()
        & order_dates[second].notna()
    )

    invalid_order = (
        valid_pair
        & (order_dates[first] < order_dates[second])
    )

    print(f"{label}: {int(invalid_order.sum())}")

print("\nExamples of delivered-before-shipped:")
mask = (
    order_dates["delivered_at"].notna()
    & order_dates["shipped_at"].notna()
    & (
        order_dates["delivered_at"]
        < order_dates["shipped_at"]
    )
)

display(
    orders.loc[
        mask,
        ["id", "status", "placed_at", "paid_at",
         "shipped_at", "delivered_at"],
    ].head(10)
)


DATE PARSING
----------------------------------------
placed_at: invalid dates = 0
paid_at: invalid dates = 0
shipped_at: invalid dates = 0
delivered_at: invalid dates = 0

CHRONOLOGICAL VALIDATION
----------------------------------------
Paid before order: 0
Shipped before order: 0
Delivered before order: 0
Delivered before shipped: 0
Delivered before paid: 0

Examples of delivered-before-shipped:


,id,status,placed_at,paid_at,shipped_at,delivered_at


In [73]:

# 1. Check missing product ratings against review counts

rating_missing = products["rating"].isna()

print("PRODUCT RATING VALIDATION")
print("-" * 40)
print("Products without rating:", int(rating_missing.sum()))

print(
    "Missing rating AND zero reviews:",
    int(
        (
            rating_missing
            & products["review_count"].eq(0)
        ).sum()
    ),
)

print(
    "Missing rating BUT positive review count:",
    int(
        (
            rating_missing
            & products["review_count"].gt(0)
        ).sum()
    ),
)

print(
    "Rating exists BUT zero reviews:",
    int(
        (
            products["rating"].notna()
            & products["review_count"].eq(0)
        ).sum()
    ),
)

print("\nPotential inconsistencies:")
display(
    products.loc[
        (
            rating_missing
            & products["review_count"].gt(0)
        )
        | (
            products["rating"].notna()
            & products["review_count"].eq(0)
        ),
        ["id", "name", "rating", "review_count"],
    ].head(20)
)


# 2. Check compare_at_price against price

compare_available = products["compare_at_price"].notna()

print("\nCOMPARE-AT-PRICE VALIDATION")
print("-" * 40)
print("Available comparison prices:", int(compare_available.sum()))

print(
    "Compare-at-price higher than price:",
    int(
        (
            compare_available
            & (
                products["compare_at_price"]
                > products["price"]
            )
        ).sum()
    ),
)

print(
    "Compare-at-price equal to price:",
    int(
        (
            compare_available
            & (
                products["compare_at_price"]
                == products["price"]
            )
        ).sum()
    ),
)

print(
    "Compare-at-price lower than price:",
    int(
        (
            compare_available
            & (
                products["compare_at_price"]
                < products["price"]
            )
        ).sum()
    ),
)

print("\nExamples where compare-at-price is lower than price:")
display(
    products.loc[
        compare_available
        & (
            products["compare_at_price"]
            < products["price"]
        ),
        ["id", "name", "price", "compare_at_price"],
    ].head(10)
)


PRODUCT RATING VALIDATION
----------------------------------------
Products without rating: 559
Missing rating AND zero reviews: 559
Missing rating BUT positive review count: 0
Rating exists BUT zero reviews: 0

Potential inconsistencies:


,id,name,rating,review_count



COMPARE-AT-PRICE VALIDATION
----------------------------------------
Available comparison prices: 355
Compare-at-price higher than price: 355
Compare-at-price equal to price: 0
Compare-at-price lower than price: 0

Examples where compare-at-price is lower than price:


,id,name,price,compare_at_price


In [74]:

# STEP 3: DATA QUALITY INVESTIGATION

# 1. Check whether foreign keys exist in parent tables
print("=" * 60)
print("1. REFERENTIAL INTEGRITY")
print("=" * 60)

checks = [
    (
        "orders.customer_id",
        orders["customer_id"],
        customers["id"],
    ),
    (
        "order_items.order_id",
        order_items["order_id"],
        orders["id"],
    ),
    (
        "order_items.product_id",
        order_items["product_id"],
        products["id"],
    ),
    (
        "reviews.product_id",
        reviews["product_id"],
        products["id"],
    ),
    (
        "reviews.customer_id",
        reviews["customer_id"],
        customers["id"],
    ),
]

for label, child_ids, parent_ids in checks:
    invalid = ~child_ids.isin(parent_ids)

    print(f"\n{label}")
    print("Invalid references:", int(invalid.sum()))


# 2. Check date fields that should be chronological
print("\n" + "=" * 60)
print("2. ORDER DATE LOGIC")
print("=" * 60)

date_columns = [
    "placed_at",
    "paid_at",
    "shipped_at",
    "delivered_at",
]

order_dates = orders[date_columns].apply(
    pd.to_datetime, errors="coerce", utc=True
)

for column in date_columns:
    invalid_dates = (
        orders[column].notna()
        & order_dates[column].isna()
    ).sum()

    print(f"{column}: unparseable dates = {invalid_dates}")

print(
    "\nPaid before order:",
    int(
        (
            order_dates["paid_at"].notna()
            & order_dates["placed_at"].notna()
            & (
                order_dates["paid_at"]
                < order_dates["placed_at"]
            )
        ).sum()
    ),
)

print(
    "Shipped before order:",
    int(
        (
            order_dates["shipped_at"].notna()
            & order_dates["placed_at"].notna()
            & (
                order_dates["shipped_at"]
                < order_dates["placed_at"]
            )
        ).sum()
    ),
)

print(
    "Delivered before shipped:",
    int(
        (
            order_dates["delivered_at"].notna()
            & order_dates["shipped_at"].notna()
            & (
                order_dates["delivered_at"]
                < order_dates["shipped_at"]
            )
        ).sum()
    ),
)


# 3. Check order item calculations
print("\n" + "=" * 60)
print("3. ORDER ITEM CALCULATIONS")
print("=" * 60)

calculated_total = (
    order_items["quantity"] * order_items["unit_price"]
)

difference = (
    order_items["line_total"] - calculated_total
).abs()

print("Quantity <= 0:", int((order_items["quantity"] <= 0).sum()))
print("Unit price < 0:", int((order_items["unit_price"] < 0).sum()))
print("Line total < 0:", int((order_items["line_total"] < 0).sum()))
print("Line total mismatches:", int((difference > 0.01).sum()))

print("\nExample mismatches:")
display(
    order_items.loc[
        difference > 0.01,
        [
            "order_id",
            "product_id",
            "quantity",
            "unit_price",
            "line_total",
        ],
    ].head(10)
)


# 4. Check rating ranges
print("\n" + "=" * 60)
print("4. RATING VALIDATION")
print("=" * 60)

print(
    "Product ratings outside 0-5:",
    int(
        (
            products["rating"].notna()
            & ~products["rating"].between(0, 5)
        ).sum()
    ),
)

print(
    "Review ratings outside 1-5:",
    int((~reviews["rating"].between(1, 5)).sum()),
)


# 5. Check negative financial values
print("\n" + "=" * 60)
print("5. FINANCIAL VALUE VALIDATION")
print("=" * 60)

for dataset_name, df, columns in [
    (
        "orders",
        orders,
        ["subtotal", "shipping", "tax", "discount", "total"],
    ),
    (
        "products",
        products,
        ["price", "compare_at_price"],
    ),
]:
    print(f"\n{dataset_name.upper()}")

    for column in columns:
        negative_count = int((df[column] < 0).sum())
        print(f"{column}: negative values = {negative_count}")


# 6. Compare customer summary with actual orders
print("\n" + "=" * 60)
print("6. CUSTOMER ORDER SUMMARY")
print("=" * 60)

actual_order_counts = orders.groupby("customer_id").size()

customer_check = customers[["id", "orders_count"]].copy()
customer_check["actual_order_count"] = (
    customer_check["id"]
    .map(actual_order_counts)
    .fillna(0)
    .astype(int)
)

customer_check["count_matches"] = (
    customer_check["orders_count"]
    == customer_check["actual_order_count"]
)

print(
    "Customers whose orders_count differs from observed orders:",
    int((~customer_check["count_matches"]).sum()),
)

display(
    customer_check.loc[
        ~customer_check["count_matches"]
    ].head(10)
)

print("\nData quality investigation completed.")


1. REFERENTIAL INTEGRITY

orders.customer_id
Invalid references: 0

order_items.order_id
Invalid references: 0

order_items.product_id
Invalid references: 0

reviews.product_id
Invalid references: 0

reviews.customer_id
Invalid references: 0

2. ORDER DATE LOGIC
placed_at: unparseable dates = 0
paid_at: unparseable dates = 0
shipped_at: unparseable dates = 0
delivered_at: unparseable dates = 0

Paid before order: 0
Shipped before order: 0
Delivered before shipped: 0

3. ORDER ITEM CALCULATIONS
Quantity <= 0: 0
Unit price < 0: 0
Line total < 0: 0
Line total mismatches: 0

Example mismatches:


,order_id,product_id,quantity,unit_price,line_total



4. RATING VALIDATION
Product ratings outside 0-5: 0
Review ratings outside 1-5: 0

5. FINANCIAL VALUE VALIDATION

ORDERS
subtotal: negative values = 0
shipping: negative values = 0
tax: negative values = 0
discount: negative values = 0
total: negative values = 0

PRODUCTS
price: negative values = 0
compare_at_price: negative values = 0

6. CUSTOMER ORDER SUMMARY
Customers whose orders_count differs from observed orders: 234


,id,orders_count,actual_order_count,count_matches
0,1,4,6,False
4,5,3,4,False
5,6,0,1,False
7,8,2,3,False
9,10,4,5,False
14,15,2,3,False
15,16,4,5,False
21,22,3,4,False
23,24,5,7,False
32,33,3,4,False



Data quality investigation completed.


In [75]:
for name, df in datasets.items():
    print(f"\n{'=' * 50}")
    print(f"DATASET: {name.upper()}")
    print("\nMissing values:")
    print(df.isna().sum())

    print("\nDuplicate rows:", df.duplicated().sum())

    print("\nUnique values per column:")
    print(df.nunique())


DATASET: PRODUCTS

Missing values:
id                     0
created_at             0
updated_at             0
sku                    0
name                   0
slug                   0
description            0
category_id            0
brand                  0
price                  0
compare_at_price    1645
currency               0
color                  0
weight_g               0
tags                   0
in_stock               0
rating               559
review_count           0
status                 0
dtype: int64

Duplicate rows: 0

Unique values per column:
id                  2000
created_at          2000
updated_at          2000
sku                 2000
name                1643
slug                2000
description         1890
category_id            8
brand                 32
price               1975
compare_at_price     352
currency               1
color                 10
weight_g            1844
tags                 310
in_stock               2
rating                23
revie

In [76]:

from pathlib import Path
import pandas as pd
import numpy as np

# 1. Load all datasets
RAW_DIR = Path("../data/raw")

datasets = {
    "products": pd.read_csv(RAW_DIR / "products.csv"),
    "customers": pd.read_csv(RAW_DIR / "customers.csv"),
    "orders": pd.read_csv(RAW_DIR / "orders.csv"),
    "order_items": pd.read_csv(RAW_DIR / "order_items.csv"),
    "reviews": pd.read_csv(RAW_DIR / "reviews.csv"),
}

# 2. General overview: rows and columns
overview = pd.DataFrame([
    {
        "dataset": name,
        "rows": df.shape[0],
        "columns": df.shape[1],
        "memory_usage_mb": round(
            df.memory_usage(deep=True).sum() / (1024 ** 2), 3
        ),
    }
    for name, df in datasets.items()
])

print("=" * 70)
print("1. DATASET OVERVIEW")
print("=" * 70)
display(overview)

# 3. Data types of all columns
print("\n" + "=" * 70)
print("2. DATA TYPES")
print("=" * 70)

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")
    display(
        pd.DataFrame({
            "column": df.columns,
            "data_type": df.dtypes.astype(str).values,
        })
    )

# 4. Missing values: count and percentage
print("\n" + "=" * 70)
print("3. MISSING VALUES")
print("=" * 70)

for name, df in datasets.items():
    missing = pd.DataFrame({
        "missing_count": df.isna().sum(),
        "missing_percentage": (
            df.isna().mean() * 100
        ).round(2),
    })

    missing = missing[missing["missing_count"] > 0]
    missing = missing.sort_values(
        "missing_percentage", ascending=False
    )

    print(f"\n--- {name.upper()} ---")

    if missing.empty:
        print("No missing values found.")
    else:
        display(missing)

# 5. Duplicate rows
print("\n" + "=" * 70)
print("4. DUPLICATE ROWS")
print("=" * 70)

duplicate_summary = pd.DataFrame([
    {
        "dataset": name,
        "duplicate_rows": int(df.duplicated().sum()),
        "duplicate_percentage": round(
            df.duplicated().mean() * 100, 2
        ),
    }
    for name, df in datasets.items()
])

display(duplicate_summary)

# 6. Unique values in each column
print("\n" + "=" * 70)
print("5. UNIQUE VALUES")
print("=" * 70)

for name, df in datasets.items():
    print(f"\n--- {name.upper()} ---")

    unique_summary = pd.DataFrame({
        "unique_values": df.nunique(dropna=True),
        "missing_values": df.isna().sum(),
    })

    display(unique_summary)

# 7. Duplicate IDs (primary-key candidates)
print("\n" + "=" * 70)
print("6. DUPLICATE ID CHECK")
print("=" * 70)

for name, df in datasets.items():
    if "id" in df.columns:
        duplicate_ids = int(
            df["id"].duplicated(keep=False).sum()
        )

        print(
            f"{name}: {duplicate_ids} rows have an ID "
            "that appears more than once."
        )

print("\nInitial data audit completed.")


1. DATASET OVERVIEW


,dataset,rows,columns,memory_usage_mb
0,products,2000,19,1.629
1,customers,800,13,0.504
2,orders,3000,21,2.872
3,order_items,6219,10,1.923
4,reviews,2500,10,0.963



2. DATA TYPES

--- PRODUCTS ---


,column,data_type
0,id,int64
1,created_at,str
2,updated_at,str
3,sku,str
4,name,str
5,slug,str
6,description,str
7,category_id,int64
8,brand,str
9,price,float64



--- CUSTOMERS ---


...


--- ORDER_ITEMS ---


,unique_values,missing_values
id,6219,0
created_at,2998,0
updated_at,2998,0
order_id,3000,0
product_id,1733,0
sku,1733,0
name,1448,0
quantity,4,0
unit_price,1713,0
line_total,3029,0



--- REVIEWS ---


,unique_values,missing_values
id,2500,0
created_at,2500,0
updated_at,2500,0
product_id,1441,0
customer_id,768,0
rating,5,0
title,12,0
body,956,0
verified_purchase,2,0
helpful_votes,41,0



6. DUPLICATE ID CHECK
products: 0 rows have an ID that appears more than once.
customers: 0 rows have an ID that appears more than once.
orders: 0 rows have an ID that appears more than once.
order_items: 0 rows have an ID that appears more than once.
reviews: 0 rows have an ID that appears more than once.

Initial data audit completed.
